In [ ]:
import os
import sys
from pathlib import Path

# Insert project root so we can import src
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

# Import reusable components from Task 2
from src.features.build_features import load_data, clean_data, build_pipeline, TARGET


In [ ]:
# load and clean data
raw_df = load_data('../data/raw/02_CortexHR_AI_ML_Dataset.xlsx')
df = clean_data(raw_df)

# Separate features (X) and target (y)
df = df.dropna(subset=[TARGET]).copy()
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print(f"Cleaned dataset: {len(df)} rows, {X.shape[1]} input columns")
print(f"Stayers (0): {(y == 0).sum()} ({(y == 0).mean():.1%})")
print(f"Leavers (1): {(y == 1).sum()} ({(y == 1).mean():.1%})")


In [ ]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training set: {len(X_train)} rows")
print(f"Locked Test set: {len(X_test)} rows")


In [ ]:
# train dummy model baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

print(f"Dummy Model Accuracy: {accuracy_score(y_test, dummy_preds):.1%}")
print(f"Dummy Model Recall:   {recall_score(y_test, dummy_preds, zero_division=0):.1%}")
print("Insight: 85.3% accuracy sounds good, but it caught 0 leavers! Accuracy is misleading.")


In [ ]:
# cross validation
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Logistic Regression (Balanced)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Decision Tree (Balanced)": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["precision", "recall", "f1", "roc_auc"]

cv_results = []
full_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ("features", build_pipeline()),
        ("classifier", model)
    ])
    full_pipelines[name] = pipe
    
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_results.append({
        "Model": name,
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "F1 Mean": scores["test_f1"].mean(),
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
    })

cv_df = pd.DataFrame(cv_results)
cv_df.round(3)


In [ ]:
# locked test set evaluation
test_results = [{
    "Model": "Dummy (Always Stays)",
    "Accuracy": accuracy_score(y_test, dummy_preds),
    "Precision": 0.0,
    "Recall": 0.0,
    "F1": 0.0,
    "ROC-AUC": np.nan
}]

test_preds = {}
test_probs = {}

for name, pipe in full_pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    test_preds[name] = preds
    test_probs[name] = probs
    
    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs),
    })

test_df = pd.DataFrame(test_results)
test_df.round(3)


In [ ]:
#visualization
best_name = "Logistic Regression (Balanced)"
best_pipe = full_pipelines[best_name]
best_pred = test_preds[best_name]
best_prob = test_probs[best_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=["Stayed (0)", "Left (1)"]).plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title(f"Confusion Matrix: {best_name}")

# ROC Curve
RocCurveDisplay.from_predictions(y_test, best_prob, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], "k--", label="Random Chance (0.50)")
axes[1].set_title(f"ROC Curve: {best_name}")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
